In [19]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import random
import pandas as pd

# Load and slice shift data
shiftData = pd.read_csv('nhl_shifts.csv')
shiftData = shiftData[1:1000]

# Helper functions to convert time
def time_to_sec(t):
    m, s = map(int, t.split(':'))
    return m * 60 + s

def per_time_to_sec(period, time):
    if period == 1:
        return time_to_sec(time)
    elif period == 2:
        return 1200 + time_to_sec(time)
    elif period == 3:
        return 2400 + time_to_sec(time)
    elif period == 4:
        return 3600 + time_to_sec(time)
    else:
        raise ValueError(f"Invalid period: {period}")

# Get unique player IDs and team IDs
playerIDs = shiftData['playerID'].unique()
team_ids = shiftData['teamId'].unique()

# Convert shifts to a list of dictionaries
shifts_list = shiftData.to_dict(orient='records')

# Calculate max end time
max_end_time = max(per_time_to_sec(shift['period'], shift['endTime']) for shift in shifts_list)
print(f"Max end time: {max_end_time}")

# Initialize a list to track players on the ice at each second
players_on_ice = [[] for _ in range(max_end_time + 1)]

# Loop through shifts and populate players on the ice
for shift in shifts_list:
    converted_start = per_time_to_sec(shift['period'], shift['startTime'])
    converted_end = per_time_to_sec(shift['period'], shift['endTime'])
    
    # Add the player to all seconds between start and end of their shift
    for i in range(converted_start, converted_end):
        if 0 <= i <= max_end_time:
            players_on_ice[i].append(shift['playerID'])

# Output the list of players on the ice at each second
print(players_on_ice)





Max end time: 3600
[[8470638, 8473419, 8474596, 8474884, 8475225, 8475233, 8475279, 8476891, 8476999, 8477476, 8479325, 8480018, 8465009, 8473504, 8473575, 8474673, 8474709, 8475231, 8475852, 8476853, 8478115, 8480144], [8470638, 8473419, 8474596, 8474884, 8475225, 8475233, 8475279, 8476891, 8476999, 8477476, 8479325, 8480018, 8465009, 8473504, 8473575, 8474673, 8474709, 8475231, 8475852, 8476853, 8478115, 8480144], [8470638, 8473419, 8474596, 8474884, 8475225, 8475233, 8475279, 8476891, 8476999, 8477476, 8479325, 8480018, 8465009, 8473504, 8473575, 8474673, 8474709, 8475231, 8475852, 8476853, 8478115, 8480144], [8470638, 8473419, 8474596, 8474884, 8475225, 8475233, 8475279, 8476891, 8476999, 8477476, 8479325, 8480018, 8465009, 8473504, 8473575, 8474673, 8474709, 8475231, 8475852, 8476853, 8478115, 8480144], [8470638, 8473419, 8474596, 8474884, 8475225, 8475233, 8475279, 8476891, 8476999, 8477476, 8479325, 8480018, 8465009, 8473504, 8473575, 8474673, 8474709, 8475231, 8475852, 8476853,

In [5]:

# Assume: on_ice = [{team_id1: set(...), team_id2: set(...)}, ...]
# Ensure that team ids are integers and not np.int64

rows = []

# Get consistent team ordering (assuming 2 teams per game)
team_ids = list(on_ice[0].keys())  # assuming both teams always present

team1, team2 = int(team_ids[0]), int(team_ids[1])  # ensure team IDs are integers

for second, second_data in enumerate(on_ice):
    # Ensure each team is converted to an integer key
    team1_players = ','.join(map(str, second_data[int(team1)]))
    team2_players = ','.join(map(str, second_data[int(team2)]))

    row = {
        'second': second,
        f'team_{team1}': team1_players,
        f'team_{team2}': team2_players,
    }
    rows.append(row)

# Convert to DataFrame
df = pd.DataFrame(rows)

# Export to CSV
df.to_csv('on_ice_by_second.csv', index=False)


In [7]:
df

,second,team_6,team_8
0,0,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
1,1,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
2,2,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
3,3,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
4,4,"8475745,8478498,8476999,8473419,8470638,847528...","8475233,8474596,8474884,8477476,8477989,847847..."
...,...,...,...
1196,1196,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1197,1197,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1198,1198,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."
1199,1199,"8480001,8475745,8478498,8477956,8476999,847341...","8475233,8478915,8477476,8477989,8478470,847350..."


In [12]:
df['team_6'][0]

'8475745,8478498,8476999,8473419,8470638,8475287,8475225,8476891,8479325'

In [10]:
count = df[df['team_6'].apply(lambda x: '8478498' in x)].count()
print(count)

second    804
team_6    804
team_8    804
dtype: int64
